In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)
# ========== PARAMS ==========
base_dir = VIDEO_DIR  # dossier racine à scanner
excel_suffixes = {".xlsx"}                                            # types d'Excel
markers = ["NOSE", "LEFT_WRIST", "RIGHT_WRIST"]                       # marqueurs à calculer
# ============================

def find_excels(root: Path):
    return [p for p in root.rglob("*") if p.suffix.lower() in excel_suffixes]

def pick_pose_sheet(xls: pd.ExcelFile) -> str:
    lower = {s.lower(): s for s in xls.sheet_names}
    for cand in ("pose", "holistic", "data", "timeseries"):
        if cand in lower:
            return lower[cand]
    # fallback: 1re feuille
    return xls.sheet_names[0]

def motion_sum_for_marker(df: pd.DataFrame, name: str) -> float:
    """Somme des déplacements euclidiens frame->frame pour un marqueur (x,y,z), insensible à la casse."""
    lc = {c.lower(): c for c in df.columns}
    try:
        cx, cy, cz = lc[f"{name.lower()}_x"], lc[f"{name.lower()}_y"], lc[f"{name.lower()}_z"]
    except KeyError:
        return np.nan  # colonnes manquantes
    # cast numeric (au cas où Excel a des strings)
    x = pd.to_numeric(df[cx], errors="coerce")
    y = pd.to_numeric(df[cy], errors="coerce")
    z = pd.to_numeric(df[cz], errors="coerce")
    step = np.sqrt(x.diff()**2 + y.diff()**2 + z.diff()**2)
    return float(step.fillna(0).sum())

# ---- Pass 1: recensement des fichiers + nombre de lignes de la feuille pose ----
files = find_excels(base_dir)
print(f"Fichiers Excel trouvés: {len(files)}")

meta_list = []
pose_cache = {}   # cache DataFrames pour éviter de relire 2x
min_len = None

for xl_path in files:
    try:
        xls = pd.ExcelFile(xl_path)
        pose_sheet = pick_pose_sheet(xls)
        df_pose = pd.read_excel(xl_path, sheet_name=pose_sheet)
        pose_cache[xl_path] = (pose_sheet, df_pose)
        n_rows = len(df_pose)
        meta_list.append({"file": str(xl_path), "pose_sheet": pose_sheet, "n_rows_original": n_rows})
        if n_rows > 1:
            min_len = n_rows if min_len is None else min(min_len, n_rows)
    except Exception as e:
        meta_list.append({"file": str(xl_path), "pose_sheet": None, "n_rows_original": np.nan, "error": str(e)})

meta_df = pd.DataFrame(meta_list).sort_values("file").reset_index(drop=True)
print("Min frames commun (min_len) =", min_len)

if min_len is None or np.isnan(min_len):
    raise RuntimeError("Aucun fichier valide avec au moins 2 lignes trouvé.")

# ---- Pass 2: calculs tronqués à min_len ----
rows = []
for xl_path in files:
    row = {
        "file": str(xl_path),
        "video_name": xl_path.stem,
        "n_used": int(min_len),
    }
    if xl_path in pose_cache:
        pose_sheet, df_pose = pose_cache[xl_path]
        row["pose_sheet"] = pose_sheet
        row["n_rows_original"] = len(df_pose)
        # tronquer
        df_used = df_pose.iloc[:min_len].reset_index(drop=True)
        # calcul par marqueur
        for m in markers:
            total = motion_sum_for_marker(df_used, m)
            row[f"total_motion_{m}"] = total
    else:
        row["pose_sheet"] = None
        row["n_rows_original"] = np.nan
        for m in markers:
            row[f"total_motion_{m}"] = np.nan
    rows.append(row)

summary = pd.DataFrame(rows).sort_values("video_name").reset_index(drop=True)

# Enrichissement: frames supprimées
summary["frames_dropped"] = summary["n_rows_original"] - summary["n_used"]

# ---- Sauvegardes ----
out_csv = EXCEL_DIR / "Summary_Motion_Nose_Wrists_equalizedFrames.csv"
out_xlsx = EXCEL_DIR / "Summary_Motion_Nose_Wrists_equalizedFrames.xlsx"
summary.to_csv(out_csv, index=False)
with pd.ExcelWriter(out_xlsx) as xl:
    summary.to_excel(xl, sheet_name="summary", index=False)
    meta_df.to_excel(xl, sheet_name="files_meta", index=False)

print("✅ Résumé créé :")
print(" -", out_csv)
print(" -", out_xlsx)

summary.head(12)

Fichiers Excel trouvés: 123
Min frames commun (min_len) = 120
✅ Résumé créé :
 - /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Summary_Motion_Nose_Wrists_equalizedFrames.csv
 - /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Summary_Motion_Nose_Wrists_equalizedFrames.xlsx


,file,video_name,n_used,pose_sheet,n_rows_original,total_motion_NOSE,total_motion_LEFT_WRIST,total_motion_RIGHT_WRIST,frames_dropped
0,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD01_P1_pose,120,Sheet1,4499,0.259788,0.235832,0.217756,4379
1,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD01_P2_pose,120,Sheet1,4490,0.270618,0.119592,0.215666,4370
2,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD02_P1_pose,120,Sheet1,4512,0.267112,0.328857,0.245218,4392
3,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD02_P2_pose,120,Sheet1,4501,0.367285,0.164280,0.455524,4381
4,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD03_P1_pose,120,Sheet1,4502,0.362678,0.440879,0.461042,4382
5,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD03_P2_pose,120,Sheet1,4511,0.273003,0.192749,0.423383,4391
6,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD04_P1_pose,120,Sheet1,4502,0.774903,1.110175,0.907535,4382
7,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD04_P2_pose,120,Sheet1,4500,0.281592,0.203964,0.325816,4380
8,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD05_P1_pose,120,Sheet1,4500,0.316100,0.671989,0.305478,4380
9,/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Vi...,SEATEDD05_P2_pose,120,Sheet1,4505,0.147199,0.083358,0.103912,4385
